In [163]:
#import libs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [164]:
df=pd.read_csv('taxi_trip_pricing.csv')
df.shape

(1000, 11)

In [165]:
#pre-processing
df.isna().sum()

Trip_Distance_km         50
Time_of_Day              50
Day_of_Week              50
Passenger_Count          50
Traffic_Conditions       50
Weather                  50
Base_Fare                50
Per_Km_Rate              50
Per_Minute_Rate          50
Trip_Duration_Minutes    50
Trip_Price               49
dtype: int64

In [166]:
df2=df.dropna()
df2.shape


(562, 11)

In [167]:
df2.head()

,Trip_Distance_km,Time_of_Day,Day_of_Week,Passenger_Count,Traffic_Conditions,Weather,Base_Fare,Per_Km_Rate,Per_Minute_Rate,Trip_Duration_Minutes,Trip_Price
0,19.35,Morning,Weekday,3.0,Low,Clear,3.56,0.80,0.32,53.82,36.2624
2,36.87,Evening,Weekend,1.0,High,Clear,2.70,1.21,0.15,37.27,52.9032
5,8.64,Afternoon,Weekend,2.0,Medium,Clear,2.55,1.71,0.48,89.33,60.2028
12,41.79,Night,Weekend,3.0,High,Clear,4.60,1.77,0.11,86.95,88.1328
14,9.91,Evening,Weekday,2.0,High,Clear,2.32,1.26,0.34,41.72,28.9914


In [168]:
#encode categorical variables
from sklearn.preprocessing import LabelEncoder

time_of_day_encoder = LabelEncoder()
time_of_day_encoder.fit(df2['Time_of_Day'])
df2['Time_of_Day'] = time_of_day_encoder.transform(df2['Time_of_Day'])

#day of week
day_of_week_encoder = LabelEncoder()
day_of_week_encoder.fit(df2['Day_of_Week'])
df2['Day_of_Week'] = day_of_week_encoder.transform(df2['Day_of_Week'])

#traffic conditions
traffic_conditions_encoder = LabelEncoder()
traffic_conditions_encoder.fit(df2['Traffic_Conditions'])
df2['Traffic_Conditions'] = traffic_conditions_encoder.transform(df2['Traffic_Conditions'])

#weather conditions
weather_conditions_encoder = LabelEncoder()
weather_conditions_encoder.fit(df2['Weather'])
df2['Weather'] = weather_conditions_encoder.transform(df2['Weather'])


/var/folders/5g/7m5tb_kn729bgtcb25k58dlc0000gn/T/ipykernel_80794/3804582008.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['Time_of_Day'] = time_of_day_encoder.transform(df2['Time_of_Day'])
/var/folders/5g/7m5tb_kn729bgtcb25k58dlc0000gn/T/ipykernel_80794/3804582008.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['Day_of_Week'] = day_of_week_encoder.transform(df2['Day_of_Week'])
/var/folders/5g/7m5tb_kn729bgtcb25k58dlc0000gn/T/ipykernel_80794/3804582008.py:16: SettingWithCopyWarning: 
A val

In [169]:
df2.head()

,Trip_Distance_km,Time_of_Day,Day_of_Week,Passenger_Count,Traffic_Conditions,Weather,Base_Fare,Per_Km_Rate,Per_Minute_Rate,Trip_Duration_Minutes,Trip_Price
0,19.35,2,0,3.0,1,0,3.56,0.80,0.32,53.82,36.2624
2,36.87,1,1,1.0,0,0,2.70,1.21,0.15,37.27,52.9032
5,8.64,0,1,2.0,2,0,2.55,1.71,0.48,89.33,60.2028
12,41.79,3,1,3.0,0,0,4.60,1.77,0.11,86.95,88.1328
14,9.91,1,0,2.0,0,0,2.32,1.26,0.34,41.72,28.9914


Goal: Train a regression model to predict the price based on the features in the dataset

In [170]:
df2.describe()

,Trip_Distance_km,Time_of_Day,Day_of_Week,Passenger_Count,Traffic_Conditions,Weather,Base_Fare,Per_Km_Rate,Per_Minute_Rate,Trip_Duration_Minutes,Trip_Price
count,562.000000,562.000000,562.000000,562.000000,562.000000,562.000000,562.000000,562.000000,562.000000,562.000000,562.000000
mean,27.772941,1.104982,0.322064,2.533808,1.227758,0.387900,3.509893,1.219858,0.288221,61.825089,57.663525
std,21.153175,1.046858,0.467684,1.108915,0.749149,0.622567,0.871082,0.430351,0.114834,32.128436,43.958741
min,1.270000,0.000000,0.000000,1.000000,0.000000,0.000000,2.010000,0.500000,0.100000,5.010000,6.126900
25%,13.135000,0.000000,0.000000,2.000000,1.000000,0.000000,2.722500,0.840000,0.190000,36.530000,33.583875
50%,26.420000,1.000000,0.000000,3.000000,1.000000,0.000000,3.545000,1.200000,0.280000,61.210000,50.157850
75%,38.827500,2.000000,1.000000,4.000000,2.000000,1.000000,4.260000,1.580000,0.387500,88.435000,69.146575
max,146.067047,3.000000,1.000000,4.000000,2.000000,2.000000,5.000000,2.000000,0.500000,119.840000,332.043689


In [171]:
y=df2['Trip_Price'] #label
x=df2.drop(['Trip_Price'],axis=1) #all features

# y.head()

In [172]:
x.shape

(562, 10)

In [173]:
#split the data 
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.25,random_state=42)

In [174]:
x_train.shape

(421, 10)

In [175]:
#select your models
#linear regression
#random forest regressor
#polynomial regression

In [176]:
#linear regression
from sklearn.linear_model import LinearRegression
lr_model=LinearRegression()

#train
lr_model.fit(x_train,y_train)



,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [177]:
#performance metrics
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

lr_predictions=lr_model.predict(x_test)
lr_mse=mean_squared_error(y_test, lr_predictions)
lr_rmse=np.sqrt(lr_mse)
lr_r2=r2_score(y_test, lr_predictions)
lr_mae=mean_absolute_error(y_test, lr_predictions)

print("Linear Regression Performance Metrics:")
print("Mean Squared Error (MSE):", lr_mse)
print("Root Mean Squared Error (RMSE):", lr_rmse)
print("R-squared (R2):", lr_r2)
print("Mean Absolute Error (MAE):", lr_mae)

Linear Regression Performance Metrics:
Mean Squared Error (MSE): 254.14965032187837
Root Mean Squared Error (RMSE): 15.94207170733711
R-squared (R2): 0.8971344136479512
Mean Absolute Error (MAE): 9.459388647870906


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


In [178]:
#random forest regressor
from sklearn.ensemble import RandomForestRegressor

rf_model=RandomForestRegressor(random_state=42)
rf_model.fit(x_train,y_train)

rf_predictions=rf_model.predict(x_test)

rf_mse=mean_squared_error(y_test, rf_predictions)
rf_rmse=np.sqrt(rf_mse)
rf_r2=r2_score(y_test, rf_predictions)
rf_mae=mean_absolute_error(y_test, rf_predictions)

print("\nRandom Forest Regressor Performance Metrics:")
print("Mean Squared Error (MSE):", rf_mse)
print("Root Mean Squared Error (RMSE):", rf_rmse)
print("R-squared (R2):", rf_r2)
print("Mean Absolute Error (MAE):", rf_mae)


Random Forest Regressor Performance Metrics:
Mean Squared Error (MSE): 112.0615223882722
Root Mean Squared Error (RMSE): 10.585911504838505
R-squared (R2): 0.954643753420972
Mean Absolute Error (MAE): 5.404562493965426


In [179]:
#multinomial regression
from sklearn.linear_model import Ridge
ridge_model=Ridge(alpha=1.0)
ridge_model.fit(x_train,y_train)

ridge_predictions=ridge_model.predict(x_test)

ridge_mse=mean_squared_error(y_test, ridge_predictions)
ridge_rmse=np.sqrt(ridge_mse)
ridge_r2=r2_score(y_test, ridge_predictions)
ridge_mae=mean_absolute_error(y_test, ridge_predictions)

print("\nRidge Regression Performance Metrics:")
print("Mean Squared Error (MSE):", ridge_mse)
print("Root Mean Squared Error (RMSE):", ridge_rmse)
print("R-squared (R2):", ridge_r2)
print("Mean Absolute Error (MAE):", ridge_mae)



Ridge Regression Performance Metrics:
Mean Squared Error (MSE): 254.75247491127905
Root Mean Squared Error (RMSE): 15.960967229816589
R-squared (R2): 0.8968904239167924
Mean Absolute Error (MAE): 9.557788305873272


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-p

In [180]:
#chose rf regressor
#tune hyperparameters
param_grid = {
    'n_estimators': [30,40,50,70],
    'max_depth': [6,8,9,10],
    'min_samples_split': [2, 3,4],
    'min_samples_leaf': [2,3,4]
}

#k-fold cross-validation
from sklearn.model_selection import GridSearchCV
rf_model = RandomForestRegressor(random_state=42)

grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=5, scoring='neg_mean_absolute_percentage_error', n_jobs=-1)
grid_search.fit(x_train, y_train)

print("Best Hyperparameters:", grid_search.best_params_)
print("Best Score (MAPE):", -grid_search.best_score_)

best_model = grid_search.best_estimator_
best_metrics = grid_search.best_score_
mae=mean_absolute_error(y_test, best_model.predict(x_test))
print("Mean Absolute Error (MAE) on Test Set:", mae)
mse=mean_squared_error(y_test, best_model.predict(x_test))
rmse=np.sqrt(mse)
print("Mean Squared Error (MSE) on Test Set:", mse)
print("Root Mean Squared Error (RMSE) on Test Set:", rmse)


Best Hyperparameters: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 50}
Best Score (MAPE): 0.11142095953394875
Mean Absolute Error (MAE) on Test Set: 5.309143645590853
Mean Squared Error (MSE) on Test Set: 104.10845020472456
Root Mean Squared Error (RMSE) on Test Set: 10.20335485047563


In [193]:
#prediction
input=[[19.35, 2, 0, 3.0, 1, 0, 3.56, 0.80, 0.32, 53.82]]
prediction=best_model.predict(input)
print("Prediction:", prediction)

Prediction: [36.94321584]


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [199]:
encoded_Weather=weather_conditions_encoder.transform(['Clear'])


inputX=[[19.35, 2, 0, 3.0, 1, encoded_Weather[0], 3.56, 0.80, 0.32, 53.82]]
prediction=best_model.predict(inputX)
print("Prediction:", prediction)

Prediction: [36.94321584]


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
